In [ ]:
# Task 3 PCA

import pandas as pd

file_path = r"../data/housing_data.csv"

df = pd.read_csv(file_path)

In [ ]:
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

# List of selected continuous variables from Part D1
continuous_vars = [
    'SquareFootage', 'NumBathrooms', 'NumBedrooms', 'BackyardSpace',
    'CrimeRate', 'SchoolRating', 'AgeOfHome', 'DistanceToCityCenter',
    'RenovationQuality', 'LocalAmenities', 'TransportAccess',
    'PreviousSalePrice', 'Windows'
]

# Extract the data to be standardized
X_continuous = df[continuous_vars]

# Standardize using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_continuous)

# Create cleaned DataFrame with standardized values
df_cleaned = pd.DataFrame(X_scaled, columns=continuous_vars)

# Save to CSV
df_cleaned.to_csv("standardized_cleaned_data.csv", index=False)

# Display the cleaned standardized dataset
df_cleaned.head()

In [ ]:
# Add the dependent variable back (not standardized for descriptive stats)
df_descriptive = df[["Price"] + continuous_vars]

# Show descriptive statistics
df_descriptive.describe().T

In [ ]:
mode_values = df_descriptive.mode().iloc[0]
mode_values.to_frame(name='mode')

In [ ]:
from sklearn.decomposition import PCA

# Run PCA on standardized data
pca = PCA()  # No n_components specified: will return all components
principal_components = pca.fit_transform(df_cleaned)

# Create DataFrame of all principal components
pc_columns = [f'PC{i+1}' for i in range(principal_components.shape[1])]
df_pca = pd.DataFrame(principal_components, columns=pc_columns)

# Display first few rows of the full PCA matrix
df_pca.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Explained variance per component
explained_variance = pca.explained_variance_ratio_
eigenvalues = pca.explained_variance_

# Scree plot
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance)+1), explained_variance, marker='o', linestyle='-')
plt.title('Scree Plot (Explained Variance by Component)')
plt.xlabel('Principal Component')
plt.ylabel('Proportion of Variance Explained')
plt.grid(True)
plt.xticks(range(1, len(explained_variance)+1))
plt.tight_layout()
plt.show()

In [ ]:
eigenvalues

In [ ]:
# Get the explained variance (proportion) for the first 4 components
for i, var in enumerate(explained_variance[:4], 1):
    print(f"PC{i}: {var:.4f} ({var * 100:.2f}%)")

#### #Linear Regression

In [ ]:
# Data Splitting

from sklearn.model_selection import train_test_split

# Select only the 4 retained principal components
X_pca_selected = df_pca[['PC1', 'PC2', 'PC3', 'PC4']]

# Use original 'Price' column as the target variable (not standardized)
y = df['Price']

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X_pca_selected, y, test_size=0.2, random_state=42)

# Combine features and target into training and test DataFrames
train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

# Save to CSV files
train_df.to_csv("pca_train_dataset.csv", index=False)
test_df.to_csv("pca_test_dataset.csv", index=False)

# Display head of train set
train_df.head()

In [ ]:
#Initial Modeling with OLS Regression

import statsmodels.api as sm

# Add constant for intercept
X_train_const = sm.add_constant(X_train)

# Build initial OLS regression model
model = sm.OLS(y_train, X_train_const).fit()

# View full summary
print(model.summary())

In [ ]:
# Optimization of the model

# Drop PC4
X_train_opt = X_train.drop(columns=['PC4'])
X_train_opt_const = sm.add_constant(X_train_opt)

# Refit the model
optimized_model = sm.OLS(y_train, X_train_opt_const).fit()
print(optimized_model.summary())

In [ ]:
from sklearn.metrics import mean_squared_error

# Predict on the training set
y_train_pred = optimized_model.predict(X_train_opt_const)

# Calculate MSE
mse_train = mean_squared_error(y_train, y_train_pred)
print(f"Training MSE: {mse_train:.2f}")

In [ ]:
# Prepare the test data using only PC1, PC2, PC3
X_test_opt = X_test[['PC1', 'PC2', 'PC3']]
X_test_opt_const = sm.add_constant(X_test_opt)

# Predict using the optimized model
y_test_pred = optimized_model.predict(X_test_opt_const)

# Calculate MSE on test set
from sklearn.metrics import mean_squared_error
mse_test = mean_squared_error(y_test, y_test_pred)
print(f"Test MSE: {mse_test:.2f}")

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF requires a constant term
X_vif = sm.add_constant(X_train_opt)

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data